# TinyGPT — Full LLM Pipeline on GPU

Train the complete modern LLM stack from scratch:
1. **Pre-training** — Learn French from Victor Hugo novels
2. **SFT** — Learn to answer questions about Hugo's works
3. **DPO** — Learn to prefer good answers over bad ones
4. **LoRA** — Adapt to Balzac with only 1.8% of parameters

**Runtime → Change runtime type → T4 GPU** for 10-50x speedup vs CPU.

## Setup

In [ ]:
# Clone the repo
!git clone https://github.com/matt-grain/tinyGPT.git
%cd tinyGPT
!pip install torch numpy

In [ ]:
# Verify GPU is available
from tinygpt import get_device
device = get_device()

## Step 1: Pre-training

Train TinyGPT on raw Victor Hugo text (Les Misérables, Notre-Dame de Paris, Quatrevingt-treize).
The model learns next-token prediction — pure autocomplete.

**Tip:** Edit `config.json` to increase epochs for better results (30 epochs recommended).

In [ ]:
# Update config for GPU training — more epochs since it's faster
import json
from pathlib import Path

config = json.loads(Path('config.json').read_text())
config['pretrain']['epochs'] = 30
config['sft']['epochs'] = 50
config['dpo']['epochs'] = 30
config['lora']['epochs'] = 10
Path('config.json').write_text(json.dumps(config, indent=2))
print('Config updated for GPU training')

In [ ]:
!python pretrain.py

## Step 2: Supervised Fine-Tuning (SFT)

Same model, same architecture, same loss function.
Only the **data format** changes: structured Q&A pairs instead of raw text.
Loss is **masked** on the question — we only train on the answer tokens.

In [ ]:
!python sft.py

## Step 3: Direct Preference Optimization (DPO)

Teach the model to prefer good answers over bad ones.
No reward model needed (unlike full RLHF) — just preference pairs.
Uses a frozen reference model to prevent drifting.

In [ ]:
!python dpo.py

## Step 4: LoRA Adaptation

Adapt the Hugo model to Balzac by training only **1.8% of parameters**.
The adapter file is ~50KB — you could have 100 adapters for different authors.

In [ ]:
!python lora_train.py

## Interactive Generation

Play with the trained model!

In [ ]:
from tinygpt import TinyGPT, Tokenizer, generate, get_device
from tinygpt.checkpoint import load_checkpoint, auto_detect_latest
from pathlib import Path

device = get_device()
checkpoint_path = auto_detect_latest(Path('snapshots'))
model, tokenizer, hparams = load_checkpoint(checkpoint_path, device)

# Try different seeds and temperatures!
seeds = [
    'Il était une fois',
    'La cathédrale',
    'Le vieillard',
    'Paris',
]

for seed in seeds:
    print(f'\n--- Seed: "{seed}" ---')
    text = generate(
        model, tokenizer, seed,
        context_length=hparams['context_length'],
        num_words=100,
        temperature=0.8,
        device=device,
    )
    print(text)